#**BANCO DE DADOS**

##Informações

NÃO TRATADOS

**Primeira base:**

Período (ANO_APOLICE): 2006-2015 (tem um ano 2103 (observação 585.910))

N° de observações: 617.684

N° de variáveis: 36

**Segunda base:**

Período (ANO_APOLICE): 2016-2024

N° de observações: 1.048.566


N° de variáveis: 38


**Estou mexendo apenas no segundo banco de dados**

##Primeiro tratamento dos dados

- Remover a linha com o ano incorretor 2103.
- Deixar apenas as variáveis pertinentes.
- Unir as bases em ordem cronológica.

In [ ]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.4/87.4 kB 3.7 MB/s eta 0:00:00


In [ ]:
##BIBLIOTECAS

import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import skew
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt

import seaborn as sns
import category_encoders as ce
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, mean_squared_error
import shap
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [ ]:

# =========================
# 1. Caminhos dos arquivos
# =========================
#arquivo_base1 = "/content/drive/MyDrive/TCC 2/BANCO DE DADOS/psrdadosabertos2006a2015excel.xlsx"
arquivo_base2 = "/content/drive/MyDrive/TCC 2/BANCO DE DADOS/dados_abertos_psr_2016a2024.xlsx"

# =========================
# 2. Leitura das bases
# =========================
#base1 = pd.read_excel(arquivo_base1)
base2 = pd.read_excel(arquivo_base2)

# =========================
# 3. Remover ano incorreto (2103) da primeira base
# =========================
#base1 = base1[base1["ANO_APOLICE"] != 2103]

# =========================
# 4. Variáveis pertinentes
# =========================
variaveis_selecionadas = [
    "ANO_APOLICE",
    "NM_MUNICIPIO_PROPRIEDADE",
    "SG_UF_PROPRIEDADE",
    "NM_CLASSIF_PRODUTO",
    "NM_CULTURA_GLOBAL",
    "NR_AREA_TOTAL",
    "NR_ANIMAL",
    "NR_PRODUTIVIDADE_ESTIMADA",
    "NR_PRODUTIVIDADE_SEGURADA",
    "NivelDeCobertura",
    "VL_LIMITE_GARANTIA",
    "VL_PREMIO_LIQUIDO",
    "PE_TAXA",
    "VL_SUBVENCAO_FEDERAL",
    "VALOR_INDENIZAÇÃO",
    "EVENTO_PREPONDERANTE"
]

# =========================
# 5. Manter apenas as variáveis desejadas
# =========================
#base1 = base1[variaveis_selecionadas]
base2 = base2[variaveis_selecionadas]

# =========================
# 6. Unir as bases
# =========================
base_final = pd.concat([ base2], ignore_index=True)

# =========================
# 7. Ordenar cronologicamente
# =========================
base_final = base_final.sort_values(by="ANO_APOLICE")

# =========================
# 8. Exportar base final
# =========================
#base_final.to_csv(
    #"base_unificada_2006_2024.csv",
    #index=False,
    #sep=";",
    #encoding="utf-8-sig"
#)

In [ ]:
base_final.info()

In [ ]:
# Estatística descritiva das variáveis numéricas
desc_stats = base_final.describe(
    include=[np.number],  # apenas variáveis numéricas
    percentiles=[0.25, 0.5, 0.75]
).T  # transpor para ficar em formato de tabela

# Renomear colunas para ficar mais claro
desc_stats = desc_stats.rename(columns={
    "mean": "Média",
    "std": "Desvio Padrão",
    "min": "Mínimo",
    "25%": "1º Quartil (25%)",
    "50%": "Mediana (50%)",
    "75%": "3º Quartil (75%)",
    "max": "Máximo"
})

print(desc_stats)


In [ ]:
# Proporção de valores ausentes por variável
missing = base_final.isnull().mean().sort_values(ascending=False) * 100

# Transformar em tabela
missing_table = missing.reset_index()
missing_table.columns = ["Variável", "Percentual de Valores Ausentes"]

print(missing_table)


## **CORREÇÃO DO BANCO DE DADOS**

### EXPLICAÇÃO

- No código abaixo foi substituido os - do banco de dados por NA.

- Depois foi verificado que algumas variáveis numéricas eram tratadas como object e foram modificados para float e trocado o separador de decimal por ponto.

- Foi verificado um alto índice de valores ausentes que foi tratado pela mediana e moda.

- A variável VALOR_INDENIZAÇÃO tem linhas com '-' isso significa que se o evento (sinistro) não ocorreu então não houve pagamento. Com isso no lugar de '-' será colocado 0.

- No mesmo caso se encontra o EVENTO_PREPONDERANTE, nele recebera SEM_SINISTRO.

- Criação da variável região.

- Criação das variáveis monetárias deflacionadas com o IPCA (site Sidra do IBGE, tabela 1737)

In [ ]:
# ===========================
# Substituir traços por NaN
# ===========================
base_final.replace("-", pd.NA, inplace=True)

# =======================================================================
# Lista de variáveis numéricas que precisam ser convertidas para float
# =======================================================================
colunas_numericas_convertidas = [
    'NR_AREA_TOTAL',
    'NR_ANIMAL',
    'NR_PRODUTIVIDADE_ESTIMADA',
    'NR_PRODUTIVIDADE_SEGURADA',
    'VL_LIMITE_GARANTIA',
    'VL_PREMIO_LIQUIDO',
    'PE_TAXA',
    'VL_SUBVENCAO_FEDERAL',
    'VALOR_INDENIZAÇÃO'
]

# NivelDeCobertura percentual (ex.: 70, 80, 90):
colunas_numericas_convertidas.append('NivelDeCobertura')

# ============================================================================
# Converter colunas numéricas que estão como texto (vírgula -> ponto decimal)
# ============================================================================
for col in colunas_numericas_convertidas:
    base_final[col] = base_final[col].astype(str).str.replace(",", ".")
    base_final[col] = pd.to_numeric(base_final[col], errors='coerce')

# ===========================
# Imputação de valores
# ===========================
# VALOR_INDENIZAÇÃO → substituir ausentes por 0 (sem sinistro)
base_final['VALOR_INDENIZAÇÃO'] = base_final['VALOR_INDENIZAÇÃO'].fillna(0)

# NR_ANIMAL com mediana
mediana_animal = base_final['NR_ANIMAL'].median()
base_final['NR_ANIMAL'] = base_final['NR_ANIMAL'].fillna(mediana_animal)

# EVENTO_PREPONDERANTE → "SEM_SINISTRO" quando não houve sinistro
base_final['EVENTO_PREPONDERANTE'] = base_final['EVENTO_PREPONDERANTE'].fillna("SEM_SINISTRO")

# Demais variáveis numéricas com mediana (exceto VALOR_INDENIZAÇÃO já tratada)
colunas_numericas = base_final.select_dtypes(include=['float64', 'int64']).columns
for col in colunas_numericas:
    if col != 'VALOR_INDENIZAÇÃO':
        base_final[col] = base_final[col].fillna(base_final[col].median())

# Demais variáveis categóricas com moda (exceto EVENTO_PREPONDERANTE já tratada)
colunas_categoricas = base_final.select_dtypes(include=['object', 'category']).columns
for col in colunas_categoricas:
    if col != 'EVENTO_PREPONDERANTE':
        base_final[col] = base_final[col].fillna(base_final[col].mode()[0])

# ===========================
# Criar variável REGIAO
# ===========================
mapa_regioes = {
    'AC': 'Norte', 'AP': 'Norte', 'AM': 'Norte', 'PA': 'Norte', 'RO': 'Norte', 'RR': 'Norte', 'TO': 'Norte',
    'AL': 'Nordeste', 'BA': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste', 'PB': 'Nordeste', 'PE': 'Nordeste',
    'PI': 'Nordeste', 'RN': 'Nordeste', 'SE': 'Nordeste',
    'DF': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'MT': 'Centro-Oeste', 'MS': 'Centro-Oeste',
    'ES': 'Sudeste', 'MG': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
    'PR': 'Sul', 'RS': 'Sul', 'SC': 'Sul'
}

base_final['REGIAO'] = base_final['SG_UF_PROPRIEDADE'].map(mapa_regioes)

# ===========================
# Criar colunas deflacionadas
# ===========================
# Índices IPCA
ipca_indices = {
    2016: 4775.7,
    2017: 4916.46,
    2018: 5100.61,
    2019: 5320.25,
    2020: 5560.59,
    2021: 6120.04,
    2022: 6474.09,
    2023: 6773.27,
    2024: 7100.5  # ano base
}

indice_base = ipca_indices[2024]

# Lista de colunas monetárias a deflacionar
colunas_monetarias = ['VL_LIMITE_GARANTIA', 'VL_PREMIO_LIQUIDO', 'VL_SUBVENCAO_FEDERAL', 'VALOR_INDENIZAÇÃO']

# Vetorizado
for col in colunas_monetarias:
    base_final[col + '_DEF'] = base_final[col] * (
        indice_base / base_final['ANO_APOLICE'].map(ipca_indices)
    )

# ===========================
# Converter variáveis categóricas para 'category'
# ===========================
colunas_categoricas_final = [
    'NM_MUNICIPIO_PROPRIEDADE',
    'SG_UF_PROPRIEDADE',
    'NM_CLASSIF_PRODUTO',
    'NM_CULTURA_GLOBAL',
    'EVENTO_PREPONDERANTE',
    'REGIAO'
]

for col in colunas_categoricas_final:
    base_final[col] = base_final[col].astype('category')

# ===========================
# Conferências finais
# ===========================
print(base_final.isnull().sum())   # Verificar se ainda restam valores ausentes
print(base_final.dtypes)           # Conferir tipos finais das variáveis
print(base_final[['ANO_APOLICE','VALOR_INDENIZAÇÃO','VALOR_INDENIZAÇÃO_DEF']].head())  # Conferir deflação
print(base_final[['SG_UF_PROPRIEDADE', 'REGIAO']].head())  # Conferir criação da REGIAO


In [ ]:
base_final.head()

## **BAIXA VARIÂNCIA| CORRELAÇÃO | FIV/VIF**

### Filtro de baixa variância
### Análise de correlação
### Cálculo do Fator de Inflação da Variância (FIV/VIF)


### Explicação

Os resultados mostraram alta multicolineareadade entre as variáveis e FIV alto também.

- **Baixa variância:** não houve exclusões relevantes, todas as variáveis passaram.
- **Correlação alta:**
- NR_PRODUTIVIDADE_ESTIMADA ~ NR_PRODUTIVIDADE_SEGURADA (correlação 0.998).

- Variáveis monetárias originais vs. deflacionadas (VL_LIMITE_GARANTIA ~ VL_LIMITE_GARANTIA_DEF, etc.).

- VALOR_INDENIZAÇÃO ~ VALOR_INDENIZAÇÃO_DEF (correlação 0.997).

- **VIF altíssimo:** várias variáveis com VIF > 100 (indicando multicolinearidade severa).

**Possível solução:**

- **Reduzir redundância:** manter apenas as variáveis desflacionadas, ja que ela vão ser usadas no modelo.

- **Produtividade:** como NR_PRODUTIVIDADE_ESTIMADA  e NR_PRODUTIVIDADE_SEGURADA são quase idênticas ficara a segurada.

- **Cobertura:** NivelDeCobertura apresentou VIF alto, vai ser tratada com PCA.

- **ANO_APOLICE:** também com VIF alto. Transformar em variável categórica (dummies).





In [ ]:
# ===========================
# Seleção de variáveis numéricas
# ===========================
colunas_numericas = base_final.select_dtypes(include=['float64', 'int64']).columns
X = base_final[colunas_numericas]

# ===========================
# 1. Filtro de baixa variância
# ===========================
selector = VarianceThreshold(threshold=0.01)  # limiar ajustável
X_lowvar = selector.fit_transform(X)
colunas_selecionadas = X.columns[selector.get_support()]
print("Variáveis após filtro de baixa variância:", list(colunas_selecionadas))

# ===========================
# 2. Matriz de correlação
# ===========================
corr_matrix = X[colunas_selecionadas].corr()
print("Matriz de correlação:\n", corr_matrix)

# Identificar pares com correlação alta (>0.8)
correlacoes_altas = []
for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if abs(corr_matrix.iloc[i, j]) > 0.8:
            correlacoes_altas.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))
print("Variáveis altamente correlacionadas:", correlacoes_altas)

# ===========================
# 3. Fator de Inflação da Variância (VIF)
# ===========================
vif_data = pd.DataFrame()
vif_data["feature"] = colunas_selecionadas
vif_data["VIF"] = [variance_inflation_factor(X[colunas_selecionadas].values, i)
                   for i in range(len(colunas_selecionadas))]
print("Valores de VIF:\n", vif_data)

# ===========================
# 4. Decisão de exclusão
# ===========================
# Critérios: variância baixa, correlação alta ou VIF > 10
variaveis_excluir = set()
variaveis_excluir.update([par[0] for par in correlacoes_altas])  # excluir uma das correlacionadas
variaveis_excluir.update(vif_data[vif_data["VIF"] > 10]["feature"].tolist())

print("Variáveis a excluir:", variaveis_excluir)



### Soluções

- Foram excluidas as variáveis:
VL_LIMITE_GARANTIA VL_PREMIO_LIQUIDO VL_SUBVENCAO_FEDERAL VALOR_INDENIZAÇÃO NR_PRODUTIVIDADE_ESTIMADA

- Com essa exclução não precisou utilizar o PCA na variável NivelDeCobertura.

- Multicolinearidade crítica entre VL_PREMIO_LIQUIDO_DEF e VL_SUBVENCAO_FEDERAL_DEF. Será excluida a variável VL_SUBVENCAO_FEDERAL_DEF


In [ ]:
# ===========================
# 1. Excluir variáveis redundantes
# ===========================
variaveis_excluir = [
    'VL_LIMITE_GARANTIA',
    'VL_PREMIO_LIQUIDO',
    'VL_SUBVENCAO_FEDERAL',
    'VALOR_INDENIZAÇÃO',
    'NR_PRODUTIVIDADE_ESTIMADA',
    'VL_SUBVENCAO_FEDERAL_DEF',
    'VL_PREMIO_LIQUIDO_DEF'
]
base_final = base_final.drop(columns=variaveis_excluir)

# ===========================
# 2. Transformar ANO_APOLICE em dummies
# ===========================
dummies_ano = pd.get_dummies(base_final['ANO_APOLICE'], prefix='ANO')
base_final = pd.concat([base_final, dummies_ano], axis=1)

# ===========================
# 3. Seleção de variáveis numéricas para correlação e VIF
# ===========================
variaveis_numericas = [
    'NR_AREA_TOTAL',
    'NR_ANIMAL',
    'NR_PRODUTIVIDADE_SEGURADA',
    'NivelDeCobertura',
    'VL_LIMITE_GARANTIA_DEF'
]

# ===========================
# 4. Matriz de correlação
# ===========================
corr_matrix = base_final[variaveis_numericas].corr()
print("Matriz de correlação:\n", corr_matrix)

# Identificar pares com correlação alta (>0.8)
correlacoes_altas = []
for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if abs(corr_matrix.iloc[i, j]) > 0.8:
            correlacoes_altas.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))
print("Variáveis altamente correlacionadas:", correlacoes_altas)

# ===========================
# 5. VIF (Fator de Inflação da Variância)
# ===========================
X_num = base_final[variaveis_numericas]
vif_data = pd.DataFrame()
vif_data["feature"] = X_num.columns
vif_data["VIF"] = [variance_inflation_factor(X_num.values, i) for i in range(len(X_num.columns))]
print("Valores de VIF:\n", vif_data)

# ===========================
# 6. Heatmap da matriz de correlação
# ===========================
plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0)
plt.title("Heatmap da Correlação entre Variáveis Numéricas")
plt.show()

# ===========================
# 7. Criar variável AGRICOLA vs PECUARIA
# ===========================
base_final['TIPO_ATIVIDADE'] = base_final['NM_CLASSIF_PRODUTO'].apply(
    lambda x: 'PECUARIA' if str(x).upper() == 'PECUARIA' else 'AGRICOLA'
)

# Conferir distribuição
print(base_final['TIPO_ATIVIDADE'].value_counts())

# ===========================
# 8. Estatística descritiva comparativa
# ===========================
print(base_final.groupby('TIPO_ATIVIDADE')['VALOR_INDENIZAÇÃO_DEF'].describe())

In [ ]:
base_final.head()

In [ ]:
base_final.info()

### Estatística Descritiva

In [ ]:
# ===========================
# Estatística descritiva completa
# ===========================

# Estatística descritiva das variáveis numéricas (inclui quartis 25%, 50% e 75%)
print("\n--- Estatística descritiva (numéricas) ---")
print(base_final.describe(percentiles=[0.25, 0.5, 0.75]))

# Estatística descritiva das variáveis monetárias deflacionadas
print("\n--- Estatística descritiva (monetárias deflacionadas) ---")
print(base_final[['VL_LIMITE_GARANTIA_DEF',
                  'VALOR_INDENIZAÇÃO_DEF']].describe(percentiles=[0.25, 0.5, 0.75]))

# Frequência das variáveis categóricas
print("\n--- Frequência das variáveis categóricas ---")
for col in base_final.select_dtypes(include=['object']).columns:
    print(f"\n{col}:")
    print(base_final[col].value_counts().head(10))

# Distribuição de EVENTO_PREPONDERANTE
print("\n--- Distribuição de EVENTO_PREPONDERANTE ---")
print(base_final['EVENTO_PREPONDERANTE'].value_counts())

# Frequência das variáveis booleanas (anos)
print("\n--- Frequência das variáveis booleanas (anos) ---")
for col in base_final.select_dtypes(include=['bool']).columns:
    print(f"{col}: {base_final[col].sum()} registros verdadeiros")

In [ ]:
# 1. Histograma da variável VALOR_INDENIZAÇÃO_DEF com log-transformação
plt.figure(figsize=(8,6))
sns.histplot(np.log1p(base_final['VALOR_INDENIZAÇÃO_DEF']), bins=50, kde=True, color='blue')
plt.title('Histograma da Indenização Deflacionada (log1p)')
plt.xlabel('log(1 + VALOR_INDENIZAÇÃO_DEF)')
plt.ylabel('Frequência')
plt.show()

# 2. Boxplot das variáveis monetárias deflacionadas com log-transformação
plt.figure(figsize=(10,6))
monetarias = ['VL_LIMITE_GARANTIA_DEF','VALOR_INDENIZAÇÃO_DEF']
sns.boxplot(data=np.log1p(base_final[monetarias]))
plt.title('Boxplot das variáveis monetárias deflacionadas (log1p)')
plt.xticks(range(len(monetarias)), monetarias, rotation=45)
plt.ylabel('log(1 + valor)')
plt.show()

# 3. Gráfico de barras da evolução anual das apólices
plt.figure(figsize=(10,6))
anos = ['ANO_2016','ANO_2017','ANO_2018','ANO_2019','ANO_2020','ANO_2021','ANO_2022','ANO_2023','ANO_2024']
counts = [base_final[ano].sum() for ano in anos]
sns.barplot(x=anos, y=counts, palette='viridis')
plt.title('Evolução anual das apólices (2016-2024)')
plt.xlabel('Ano')
plt.ylabel('Número de apólices')
plt.xticks(rotation=45)
plt.show()

# Salvar em Excel
base_final.to_excel(
    "base_unificada_2006_2024.xlsx",
    index=False,
    engine="openpyxl"
)

# Salvar em Excel
base_final.to_excel(
    "base_final.xlsx",
    index=False,
    engine="openpyxl"
)

# MODELOS

In [ ]:
base_co = base_final.copy()
base_co.info()

In [ ]:
base_co.head()

In [ ]:
# Excluir a coluna ANO_APOLICE
base_co = base_co.drop(columns=['ANO_APOLICE', 'ANO_2016', 'PE_TAXA'])


In [ ]:
# 1. Excluir variável resposta
X = base_co.drop(columns=['VALOR_INDENIZAÇÃO_DEF'])

# 2. Converter booleanos para inteiros
X = X.astype({col: int for col in X.select_dtypes('bool').columns})

# 3. Selecionar apenas variáveis numéricas (float64 e int64)
X_num = X.select_dtypes(include=['float64','int64'])

# 4. Adicionar constante
X_const = sm.add_constant(X_num)

# 5. Calcular VIF
vif_data = pd.DataFrame()
vif_data["feature"] = X_const.columns
vif_data["VIF"] = [variance_inflation_factor(X_const.values, i)
                   for i in range(X_const.shape[1])]

print(vif_data)

In [ ]:
base_co.info()

## NOVA BASE

In [ ]:
# ============================
# 1. Filtrar severidade > 0
# ============================
base_sev = base_co[base_co['VALOR_INDENIZAÇÃO_DEF'] > 0].copy()

# ============================
# 2. Transformações para reduzir cauda longa
# ============================
# Aplicar log(1+x) nas variáveis contínuas
for col in ['NR_AREA_TOTAL', 'NR_ANIMAL', 'NR_PRODUTIVIDADE_SEGURADA',
            'VL_LIMITE_GARANTIA_DEF']:
    base_sev[col] = np.log1p(base_sev[col])

# Truncamento apenas na resposta original
lim = base_sev['VALOR_INDENIZAÇÃO_DEF'].quantile(0.99)
base_sev['VALOR_INDENIZAÇÃO_DEF'] = np.where(
    base_sev['VALOR_INDENIZAÇÃO_DEF'] > lim,
    lim,
    base_sev['VALOR_INDENIZAÇÃO_DEF']
)

# ============================
# 3. Padronização das variáveis numéricas
# ============================
numericas = ['NR_AREA_TOTAL', 'NR_ANIMAL', 'NR_PRODUTIVIDADE_SEGURADA',
             'NivelDeCobertura', 'VL_LIMITE_GARANTIA_DEF']

scaler = StandardScaler()
base_sev[numericas] = scaler.fit_transform(base_sev[numericas])

# ============================
# 4. Target Encoding nas categóricas (com suavização)
# ============================
categoricas = ['NM_MUNICIPIO_PROPRIEDADE', 'SG_UF_PROPRIEDADE',
               'NM_CLASSIF_PRODUTO', 'NM_CULTURA_GLOBAL',
               'REGIAO', 'EVENTO_PREPONDERANTE']

y_sev = base_sev['VALOR_INDENIZAÇÃO_DEF']

encoder = ce.TargetEncoder(
    cols=categoricas,
    smoothing=10   # suavização para evitar overfitting
)

base_encoded = encoder.fit_transform(base_sev, y_sev)

# ============================
# 5. Montar base final
# ============================
X_sev = pd.concat([
    base_encoded[numericas],  # numéricas padronizadas
    base_encoded[[c for c in base_encoded.columns if c.startswith('ANO_')]],  # anos (2017–2024)
    base_encoded[categoricas],  # categóricas codificadas
], axis=1)

X_sev = sm.add_constant(X_sev).astype(float)


In [ ]:
# ============================
# Excluir a coluna SG_UF_PROPRIEDADE
# ============================
X_sev = X_sev.drop(columns=['SG_UF_PROPRIEDADE'])

# Conferir se foi removida
print("Colunas restantes em X_sev:")
print(X_sev.columns)


In [ ]:
X_sev.info()

In [ ]:
# Calcular VIF para cada variável em X_sev
vif_data = pd.DataFrame()
vif_data["feature"] = X_sev.columns
vif_data["VIF"] = [variance_inflation_factor(X_sev.values, i) for i in range(X_sev.shape[1])]

print(vif_data)

#############################
#MATRIZ DE CORRELAÇÃO
#############################

cols_num = ['NR_AREA_TOTAL','NR_ANIMAL','NR_PRODUTIVIDADE_SEGURADA',
            'NivelDeCobertura','VL_LIMITE_GARANTIA_DEF','VALOR_INDENIZAÇÃO_DEF']

# Calcular matriz de correlação
corr_matrix = base_sev[cols_num].corr()

# Plotar heatmap completo
plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, annot=True, cmap="RdBu_r", center=0,
            annot_kws={"size":10}, linewidths=0.5)
plt.title("Heatmap da Correlação entre Variáveis Numéricas", fontsize=14)
plt.show()

### ESTATÍSTICA DESCRITIVA

In [ ]:
# ============================
# Estatística Descritiva Geral
# ============================

print("Resumo estatístico com percentis (25, 50, 75):")
print(X_sev.describe(percentiles=[0.25, 0.5, 0.75], include='all'))

print("\nPercentis separados (25, 50, 75) das variáveis numéricas:")
print(X_sev.quantile([0.25, 0.5, 0.75]))

print("\nValores nulos por coluna:")
print(X_sev.isnull().sum())

In [ ]:
# ============================
# Selecionar apenas categóricas codificadas
# ============================
categoricas_only = [c for c in categoricas if c != 'SG_UF_PROPRIEDADE']

# ============================
# Boxplots (percentis visuais) - categóricas
# ============================
plt.figure(figsize=(12, 6))
sns.boxplot(data=X_sev[categoricas_only], orient="h", palette="Set2")
plt.title("Boxplots - Percentis e Outliers das Variáveis Categóricas Codificadas")
plt.xlabel("Valores")
plt.show()

# ============================
# Boxplots (percentis visuais) - variáveis monetárias
# ============================
plt.figure(figsize=(12, 6))
sns.boxplot(data=base_sev[['VL_LIMITE_GARANTIA_DEF', 'VALOR_INDENIZAÇÃO_DEF']], orient="h", palette="Set2")
plt.title("Boxplots - Limite de Garantia e Valor da Indenização")
plt.xlabel("Valores")
plt.show()

# ============================
# Distribuição (curvas KDE) - variáveis monetárias
# ============================
plt.figure(figsize=(12, 6))
for col in ['VL_LIMITE_GARANTIA_DEF', 'VALOR_INDENIZAÇÃO_DEF']:
    sns.kdeplot(base_sev[col], label=col, linewidth=1.5)
plt.title("Distribuição Geral (KDE) - Limite de Garantia e Valor da Indenização")
plt.xlabel("Valores")
plt.ylabel("Densidade")
plt.legend()
plt.show()

# ============================
# Histograma geral - variáveis monetárias
# ============================
valores = base_sev[['VL_LIMITE_GARANTIA_DEF', 'VALOR_INDENIZAÇÃO_DEF']].values.flatten()
plt.figure(figsize=(10,6))
sns.histplot(valores, bins=50, kde=True, color="steelblue")
plt.title("Histograma Geral - Limite de Garantia e Valor da Indenização")
plt.xlabel("Valores")
plt.ylabel("Frequência")
plt.show()


# MODELAGENS

In [ ]:
# Base para modelagem (float)
X_sev = sm.add_constant(X_sev).astype(float)

# Base paralela para relatórios/EDA
X_sev_info = X_sev.copy()

# Recolocar variáveis categóricas com tipo 'category'
X_sev_info['SG_UF_PROPRIEDADE'] = base_sev['SG_UF_PROPRIEDADE'].astype('category')
X_sev_info['REGIAO'] = base_sev['REGIAO'].astype('category')

# Exemplo de uso:
print("Tipos em X_sev (modelagem):")
print(X_sev.dtypes.head())

print("\nTipos em X_sev_info (EDA):")
print(X_sev_info.dtypes[['SG_UF_PROPRIEDADE','REGIAO']])

In [ ]:
X_sev.info()

In [ ]:
base_sev.info()

# Salvar em CSV
base_sev.to_csv("base_sev.csv", index=False)

# Salvar em Parquet (opcional, mais compacto e rápido)
#base_sev.to_parquet("base_sev.parquet", index=False)

# Baixar para o computador
from google.colab import files
files.download("base_sev.csv")   # ou "base_sev.parquet"

# Salvar base já codificada (numérica + resposta)
base_encoded.to_csv("base_encoded.csv", index=False)


# Baixar para o computador
from google.colab import files
files.download("base_encoded.csv")
